In [ ]:
# Imports necesarios
from pathlib import Path
from pyspark.sql import SparkSession
from md_lakehouse.common.spark import get_spark_session
from md_lakehouse.common.config import load_team_config
from md_lakehouse.common.io import read_parquet_snapshot

# Configurar paths
team_config = Path("configs/teams/team01.yaml")
base_config = Path("configs/base.yaml")
run_date = "2026-02-01"

## Paso 1: Generar Datos Bronze

Primero generamos los datos crudos usando el CLI:

```bash
python -m md_lakehouse.cli generate --team configs/teams/team01.yaml --run-date 2026-02-01
```

O con make:

```bash
make generate TEAM=configs/teams/team01.yaml RUN_DATE=2026-02-01
```


In [ ]:
# Verificar que Bronze fue generado
bronze_path = Path("data/bronze")

if bronze_path.exists():
    print("✓ Bronze layer generado")
    print(f"\nTablas Bronze:")
    for table in bronze_path.iterdir():
        if table.is_dir():
            print(f"  - {table.name}")
else:
    print("✗ Bronze layer no encontrado. Ejecuta 'make generate' primero.")

## Paso 2: Explorar Datos Bronze

Carguemos algunas tablas Bronze y exploremos su estructura:


In [ ]:
# Iniciar Spark session
spark = get_spark_session(app_name="ETL_Exploration")

# Leer tabla users
users_bronze = read_parquet_snapshot(spark, Path("data/bronze"), "users", run_date)

print(f"Usuarios Bronze: {users_bronze.count():,} filas")
print(f"\nEsquema:")
users_bronze.printSchema()
print(f"\nPrimeras 5 filas:")
users_bronze.show(5, truncate=False)

In [ ]:
# Explorar distribución de canales de adquisición
print("Distribución de canales de adquisición:")
users_bronze.groupBy("acquisition_channel").count().orderBy("count", ascending=False).show()

# Explorar distribución de edades
print("\nEstadísticas de edad:")
users_bronze.describe("age").show()

## Paso 3: Ejecutar Bronze → Silver

Ahora ejecutamos el pipeline que limpia y valida los datos:

```bash
python -m md_lakehouse.cli pipeline --team configs/teams/team01.yaml --run-date 2026-02-01
```

Este pipeline:
1. Elimina duplicados
2. Valida rangos (edad, amounts, etc.)
3. Verifica integridad referencial
4. Normaliza timestamps
5. Escribe a Silver solo si pasa todas las validaciones


In [ ]:
# Explorar tabla users en Silver
users_silver = read_parquet_snapshot(spark, Path("data/silver"), "users", run_date)

print(f"Usuarios Silver: {users_silver.count():,} filas")
print(f"\nComparing Bronze vs Silver count:")
print(f"  Bronze: {users_bronze.count():,}")
print(f"  Silver: {users_silver.count():,}")
print(f"  Diferencia: {users_bronze.count() - users_silver.count():,} (filtrados)")

## Paso 4: Explorar Gold Layer

La capa Gold contiene datos agregados y features para ML:


In [ ]:
# Leer fact_usage_daily
fact_usage = read_parquet_snapshot(spark, Path("data/gold"), "fact_usage_daily", run_date)

print(f"Fact Usage Daily: {fact_usage.count():,} filas")
fact_usage.show(10)

# Analizar uso promedio por día
print("\nUso promedio diario:")
fact_usage.select("date", "sessions", "minutes").groupBy("date").avg().orderBy("date").show()

In [ ]:
# Leer churn features
churn_features = read_parquet_snapshot(spark, Path("data/gold"), "churn_features", run_date)

print(f"Churn Features: {churn_features.count():,} filas")
print(f"\nColumnas de features:")
for col in churn_features.columns:
    print(f"  - {col}")

# Ver distribución de engagement
churn_features.describe("engagement_score", "last_30d_minutes", "tenure_days").show()

In [ ]:
# Leer churn labels
churn_labels = read_parquet_snapshot(spark, Path("data/gold"), "churn_labels", run_date)

print(f"Churn Labels: {churn_labels.count():,} filas")

# Calcular tasa de churn
churn_rate = churn_labels.filter(churn_labels.label == 1).count() / churn_labels.count()
print(f"\nTasa de churn: {churn_rate:.2%}")

# Distribución de labels
churn_labels.groupBy("label").count().show()

## Paso 5: Validación de Calidad

Verifica que las validaciones de calidad funcionaron:


In [ ]:
# Check 1: No nulls en columnas requeridas
print("Check 1: Verificar no-nulls en users.user_id")
null_count = users_silver.filter(users_silver.user_id.isNull()).count()
print(f"  Nulls encontrados: {null_count} {'✗ FAIL' if null_count > 0 else '✓ PASS'}")

# Check 2: Rango de edad
print("\nCheck 2: Verificar rango de edad (14-90)")
out_of_range = users_silver.filter((users_silver.age < 14) | (users_silver.age > 90)).count()
print(f"  Fuera de rango: {out_of_range} {'✗ FAIL' if out_of_range > 0 else '✓ PASS'}")

# Check 3: Unicidad de user_id
print("\nCheck 3: Verificar unicidad de user_id")
total = users_silver.count()
unique = users_silver.select("user_id").distinct().count()
duplicates = total - unique
print(f"  Duplicados: {duplicates} {'✗ FAIL' if duplicates > 0 else '✓ PASS'}")

In [ ]:
# Check 4: Integridad referencial payments -> users
payments_silver = read_parquet_snapshot(spark, Path("data/silver"), "payments", run_date)

orphaned = (
    payments_silver
    .select("user_id")
    .distinct()
    .join(users_silver.select("user_id"), "user_id", "left_anti")
    .count()
)

print("Check 4: Integridad referencial payments -> users")
print(f"  IDs huérfanos: {orphaned} {'✗ FAIL' if orphaned > 0 else '✓ PASS'}")

In [ ]:
# Cleanup
spark.stop()

## Resumen

✅ **Aprendiste**:
1. Generar datos Bronze
2. Ejecutar pipelines ETL
3. Validar calidad de datos
4. Explorar capas Bronze/Silver/Gold
5. Verificar integridad de datos

📝 **Próximo paso**: Notebook `03_models.ipynb` para entrenar modelos ML
